# 03 — Regime-Specific Symbolic Regression

Train one symbolic correction per ΔZ regime, select equations using natural and stress validation, and save the final regime models.

In [ ]:
!pip install -q pysr scikit-learn

In [ ]:
from pathlib import Path
import json
import pickle
import time

import numpy as np
import pandas as pd


def find_project_root():
    current = Path.cwd()
    candidates = [current, current.parent]

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
        if (candidate / "notebooks").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Processed data directory not found: {DATA_DIR}. "
        "Run 01_CO2_Data_Generation first."
    )

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

train_df = pd.read_csv(DATA_DIR / "co2_train.csv")
validation_df = pd.read_csv(DATA_DIR / "co2_validation.csv")
test_df = pd.read_csv(DATA_DIR / "co2_test.csv")
stress_validation_df = pd.read_csv(
    DATA_DIR / "co2_stress_validation.csv"
)
stress_test_df = pd.read_csv(DATA_DIR / "co2_stress_test.csv")

FEATURE_COLUMNS = ["T_reduced", "rho_reduced"]
TARGET_COLUMN = "delta_Z"

with open(MODEL_DIR / "global_symbolic_model.pkl", "rb") as file:
    global_model = pickle.load(file)

with open(MODEL_DIR / "global_selection.json", "r") as file:
    selected_global_index = json.load(file)["selected_global_index"]

selected_global_index = int(selected_global_index)


In [ ]:
from pysr import PySRRegressor


In [ ]:
REGIME_NAMES = [
    "near_ideal",
    "attraction_dominated",
    "excluded_volume_dominated"
]

regime_training_data = {}

for regime_name in REGIME_NAMES:
    subset = train_df[
        train_df["regime"] == regime_name
    ].copy()

    regime_training_data[regime_name] = {
        "X": subset[
            FEATURE_COLUMNS
        ].to_numpy(),

        "y": subset[
            TARGET_COLUMN
        ].to_numpy(),

        "count": len(subset)
    }

    print(
        regime_name,
        len(subset)
    )

In [ ]:
def create_symbolic_model(random_seed):
    return PySRRegressor(
        niterations=100,
        populations=20,
        population_size=50,

        binary_operators=[
            "+",
            "-",
            "*",
            "/"
        ],

        unary_operators=[],

        maxsize=20,
        maxdepth=10,

        model_selection="best",

        elementwise_loss=(
            "loss(prediction, target) = "
            "(prediction - target)^2"
        ),

        parsimony=0.001,

        random_state=random_seed,
        deterministic=True,
        parallelism="serial",

        verbosity=1
    )

In [ ]:
regime_models = {}

regime_seeds = {
    "near_ideal": 43,
    "attraction_dominated": 44,
    "excluded_volume_dominated": 45
}

regime_training_times = {}

for regime_name in REGIME_NAMES:
    print("\n" + "=" * 70)
    print("Training:", regime_name)
    print("=" * 70)

    model = create_symbolic_model(
        regime_seeds[regime_name]
    )

    start_time = time.time()

    model.fit(
        regime_training_data[regime_name]["X"],
        regime_training_data[regime_name]["y"],
        variable_names=[
            "T_r",
            "rho_r"
        ]
    )

    elapsed_time = time.time() - start_time

    regime_models[regime_name] = model
    regime_training_times[regime_name] = elapsed_time

    print("\nSelected equation:")
    print(model.sympy())

    print(
        f"Training time: "
        f"{elapsed_time:.2f} seconds"
    )

In [ ]:
regime_equations = pd.DataFrame([
    {
        "regime": regime_name,
        "training_count":
            regime_training_data[regime_name]["count"],

        "training_time_seconds":
            regime_training_times[regime_name],

        "selected_equation":
            str(regime_models[regime_name].sympy())
    }

    for regime_name in REGIME_NAMES
])

pd.set_option(
    "display.max_colwidth",
    None
)

regime_equations

In [ ]:
def predict_oracle_regime_models(
    models,
    dataset
):
    predictions = np.empty(
        len(dataset),
        dtype=float
    )

    for regime_name, model in models.items():
        mask = (
            dataset["regime"].to_numpy()
            == regime_name
        )

        if not np.any(mask):
            continue

        X_regime = dataset.loc[
            mask,
            FEATURE_COLUMNS
        ].to_numpy()

        predictions[mask] = model.predict(
            X_regime
        )

    return predictions

In [ ]:
def evaluate_predictions(
    dataset,
    predicted_delta_z,
    dataset_name,
    model_name
):
    true_delta_z = dataset[
        "delta_Z"
    ].to_numpy()

    ideal_pressure = dataset[
        "p_ideal_Pa"
    ].to_numpy()

    true_pressure = dataset[
        "p_real_Pa"
    ].to_numpy()

    predicted_pressure = (
        ideal_pressure
        * (1.0 + predicted_delta_z)
    )

    delta_error = (
        predicted_delta_z - true_delta_z
    )

    pressure_error = (
        np.abs(
            predicted_pressure - true_pressure
        )
        / np.abs(true_pressure)
        * 100.0
    )

    return {
        "model": model_name,
        "dataset": dataset_name,

        "delta_Z_RMSE": np.sqrt(
            np.mean(delta_error ** 2)
        ),

        "delta_Z_MAE": np.mean(
            np.abs(delta_error)
        ),

        "pressure_MAPE_percent":
            np.mean(pressure_error),

        "pressure_max_error_percent":
            np.max(pressure_error)
    }


oracle_validation_predictions = (
    predict_oracle_regime_models(
        regime_models,
        validation_df
    )
)

oracle_stress_predictions = (
    predict_oracle_regime_models(
        regime_models,
        stress_validation_df
    )
)

oracle_results = pd.DataFrame([
    evaluate_predictions(
        validation_df,
        oracle_validation_predictions,
        "natural_validation",
        "oracle_regime_specific"
    ),

    evaluate_predictions(
        stress_validation_df,
        oracle_stress_predictions,
        "stress_validation",
        "oracle_regime_specific"
    )
])

oracle_results

In [ ]:
global_comparison = (
    global_validation_results.copy()
)

global_comparison.insert(
    0,
    "model",
    "global_symbolic"
)

model_comparison = pd.concat([
    global_comparison,
    oracle_results
], ignore_index=True)

model_comparison[
    [
        "model",
        "dataset",
        "delta_Z_RMSE",
        "delta_Z_MAE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]

In [ ]:
def select_equation_by_validation(
    model,
    natural_dataset,
    stress_dataset,
    max_complexity=15
):
    rows = []

    natural_X = natural_dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    natural_y = natural_dataset[
        TARGET_COLUMN
    ].to_numpy()

    stress_X = stress_dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    stress_y = stress_dataset[
        TARGET_COLUMN
    ].to_numpy()

    for equation_index, equation_row in (
        model.equations_.iterrows()
    ):
        complexity = equation_row["complexity"]

        if complexity > max_complexity:
            continue

        natural_prediction = model.predict(
            natural_X,
            index=equation_index
        )

        stress_prediction = model.predict(
            stress_X,
            index=equation_index
        )

        natural_rmse = np.sqrt(
            np.mean(
                (
                    natural_prediction
                    - natural_y
                ) ** 2
            )
        )

        stress_rmse = np.sqrt(
            np.mean(
                (
                    stress_prediction
                    - stress_y
                ) ** 2
            )
        )

        # Natural distribution and stress region
        # receive equal importance.
        selection_score = (
            0.5 * natural_rmse
            + 0.5 * stress_rmse
        )

        rows.append({
            "equation_index": equation_index,
            "complexity": complexity,
            "natural_RMSE": natural_rmse,
            "stress_RMSE": stress_rmse,
            "selection_score": selection_score,
            "equation": equation_row["equation"]
        })

    result = pd.DataFrame(rows).sort_values(
        "selection_score"
    ).reset_index(drop=True)

    best_index = int(
        result.iloc[0]["equation_index"]
    )

    return best_index, result

In [ ]:
selected_global_index, global_selection_table = (
    select_equation_by_validation(
        global_model,
        validation_df,
        stress_validation_df
    )
)

print(
    "Selected global index:",
    selected_global_index
)

print(
    "Selected global equation:"
)

print(
    global_model.sympy(
        index=selected_global_index
    )
)

global_selection_table[
    [
        "equation_index",
        "complexity",
        "natural_RMSE",
        "stress_RMSE",
        "selection_score",
        "equation"
    ]
]

In [ ]:
selected_regime_indices = {}
regime_selection_tables = {}

for regime_name in REGIME_NAMES:
    natural_subset = validation_df[
        validation_df["regime"] == regime_name
    ]

    stress_subset = stress_validation_df[
        stress_validation_df["regime"]
        == regime_name
    ]

    selected_index, selection_table = (
        select_equation_by_validation(
            regime_models[regime_name],
            natural_subset,
            stress_subset
        )
    )

    selected_regime_indices[
        regime_name
    ] = selected_index

    regime_selection_tables[
        regime_name
    ] = selection_table

    print("\n", regime_name)
    print("Index:", selected_index)

    print(
        regime_models[regime_name].sympy(
            index=selected_index
        )
    )

In [ ]:
def predict_selected_global(
    model,
    selected_index,
    dataset
):
    X = dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    return model.predict(
        X,
        index=selected_index
    )


def predict_selected_regime_models(
    models,
    selected_indices,
    dataset
):
    predictions = np.empty(
        len(dataset),
        dtype=float
    )

    for regime_name in REGIME_NAMES:
        mask = (
            dataset["regime"].to_numpy()
            == regime_name
        )

        if not np.any(mask):
            continue

        X = dataset.loc[
            mask,
            FEATURE_COLUMNS
        ].to_numpy()

        predictions[mask] = (
            models[regime_name].predict(
                X,
                index=selected_indices[
                    regime_name
                ]
            )
        )

    return predictions

In [ ]:
updated_results = []

for dataset, dataset_name in [
    (
        validation_df,
        "natural_validation"
    ),
    (
        stress_validation_df,
        "stress_validation"
    )
]:
    global_predictions = (
        predict_selected_global(
            global_model,
            selected_global_index,
            dataset
        )
    )

    regime_predictions = (
        predict_selected_regime_models(
            regime_models,
            selected_regime_indices,
            dataset
        )
    )

    updated_results.append(
        evaluate_predictions(
            dataset,
            global_predictions,
            dataset_name,
            "global_symbolic"
        )
    )

    updated_results.append(
        evaluate_predictions(
            dataset,
            regime_predictions,
            dataset_name,
            "oracle_regime_specific"
        )
    )

updated_model_comparison = pd.DataFrame(
    updated_results
)

updated_model_comparison

In [ ]:
def create_detailed_error_table(
    dataset,
    dataset_name
):
    result = dataset.copy().reset_index(
        drop=True
    )

    global_predictions = (
        predict_selected_global(
            global_model,
            selected_global_index,
            result
        )
    )

    regime_predictions = (
        predict_selected_regime_models(
            regime_models,
            selected_regime_indices,
            result
        )
    )

    true_delta_z = result[
        "delta_Z"
    ].to_numpy()

    ideal_pressure = result[
        "p_ideal_Pa"
    ].to_numpy()

    true_pressure = result[
        "p_real_Pa"
    ].to_numpy()

    global_pressure = ideal_pressure * (
        1.0 + global_predictions
    )

    regime_pressure = ideal_pressure * (
        1.0 + regime_predictions
    )

    result["dataset"] = dataset_name

    result["global_delta_Z_prediction"] = (
        global_predictions
    )

    result["regime_delta_Z_prediction"] = (
        regime_predictions
    )

    result["global_delta_Z_abs_error"] = np.abs(
        global_predictions - true_delta_z
    )

    result["regime_delta_Z_abs_error"] = np.abs(
        regime_predictions - true_delta_z
    )

    result["global_pressure_error_percent"] = (
        np.abs(global_pressure - true_pressure)
        / np.abs(true_pressure)
        * 100.0
    )

    result["regime_pressure_error_percent"] = (
        np.abs(regime_pressure - true_pressure)
        / np.abs(true_pressure)
        * 100.0
    )

    result["regime_improvement_percent"] = (
        result["global_pressure_error_percent"]
        - result["regime_pressure_error_percent"]
    )

    result["distance_to_regime_boundary"] = (
        np.minimum(
            np.abs(
                result["delta_Z"] - EPSILON
            ),
            np.abs(
                result["delta_Z"] + EPSILON
            )
        )
    )

    return result


natural_error_table = (
    create_detailed_error_table(
        validation_df,
        "natural_validation"
    )
)

stress_error_table = (
    create_detailed_error_table(
        stress_validation_df,
        "stress_validation"
    )
)

In [ ]:
worst_natural_points = (
    natural_error_table.sort_values(
        "regime_pressure_error_percent",
        ascending=False
    )
    .head(15)
)

worst_natural_points[
    [
        "T_K",
        "rho_mol_m3",
        "T_reduced",
        "rho_reduced",
        "regime",
        "delta_Z",
        "regime_delta_Z_prediction",
        "global_pressure_error_percent",
        "regime_pressure_error_percent",
        "distance_to_regime_boundary"
    ]
]

In [ ]:
worst_stress_points = (
    stress_error_table.sort_values(
        "regime_pressure_error_percent",
        ascending=False
    )
    .head(15)
)

worst_stress_points[
    [
        "T_K",
        "rho_mol_m3",
        "T_reduced",
        "rho_reduced",
        "regime",
        "delta_Z",
        "regime_delta_Z_prediction",
        "global_pressure_error_percent",
        "regime_pressure_error_percent",
        "distance_to_regime_boundary"
    ]
]

In [ ]:
def summarize_error_tails(
    error_table
):
    rows = []

    for regime_name, subset in (
        error_table.groupby("regime")
    ):
        errors = subset[
            "regime_pressure_error_percent"
        ].to_numpy()

        global_errors = subset[
            "global_pressure_error_percent"
        ].to_numpy()

        rows.append({
            "dataset":
                subset["dataset"].iloc[0],

            "regime": regime_name,
            "count": len(subset),

            "global_mean":
                np.mean(global_errors),

            "regime_mean":
                np.mean(errors),

            "regime_median":
                np.median(errors),

            "regime_95th":
                np.percentile(errors, 95),

            "regime_99th":
                np.percentile(errors, 99),

            "regime_max":
                np.max(errors)
        })

    return pd.DataFrame(rows)


tail_error_summary = pd.concat([
    summarize_error_tails(
        natural_error_table
    ),
    summarize_error_tails(
        stress_error_table
    )
], ignore_index=True)

tail_error_summary

In [ ]:
BOUNDARY_WIDTH = 0.01

for error_table, dataset_name in [
    (
        natural_error_table,
        "natural_validation"
    ),
    (
        stress_error_table,
        "stress_validation"
    )
]:
    boundary_points = error_table[
        error_table[
            "distance_to_regime_boundary"
        ] <= BOUNDARY_WIDTH
    ]

    interior_points = error_table[
        error_table[
            "distance_to_regime_boundary"
        ] > BOUNDARY_WIDTH
    ]

    print("\n", dataset_name)

    print(
        "Boundary count:",
        len(boundary_points)
    )

    print(
        "Boundary mean error:",
        boundary_points[
            "regime_pressure_error_percent"
        ].mean()
    )

    print(
        "Interior mean error:",
        interior_points[
            "regime_pressure_error_percent"
        ].mean()
    )

In [ ]:
attraction_natural = natural_error_table[
    natural_error_table["regime"] == "attraction_dominated"
]

attraction_by_temperature = (
    attraction_natural
    .groupby("T_K")
    .agg(
        count=("regime_pressure_error_percent", "size"),
        mean_error=(
            "regime_pressure_error_percent",
            "mean"
        ),
        max_error=(
            "regime_pressure_error_percent",
            "max"
        )
    )
    .reset_index()
    .sort_values("mean_error", ascending=False)
)

attraction_by_temperature

In [ ]:
attraction_train = train_df[
    train_df["regime"] == "attraction_dominated"
]

attraction_model_expanded = PySRRegressor(
    niterations=150,
    populations=20,
    population_size=60,

    binary_operators=[
        "+",
        "-",
        "*",
        "/"
    ],

    unary_operators=[
        "square"
    ],

    maxsize=25,
    maxdepth=12,

    model_selection="best",

    elementwise_loss=(
        "loss(prediction, target) = "
        "(prediction - target)^2"
    ),

    parsimony=0.001,

    random_state=46,
    deterministic=True,
    parallelism="serial",
    verbosity=1
)

attraction_model_expanded.fit(
    attraction_train[FEATURE_COLUMNS].to_numpy(),
    attraction_train[TARGET_COLUMN].to_numpy(),
    variable_names=["T_r", "rho_r"]
)

In [ ]:
new_attraction_index, new_attraction_table = (
    select_equation_by_validation(
        attraction_model_expanded,
        validation_df[
            validation_df["regime"]
            == "attraction_dominated"
        ],
        stress_validation_df[
            stress_validation_df["regime"]
            == "attraction_dominated"
        ],
        max_complexity=25
    )
)

print("New attraction equation:")
print(
    attraction_model_expanded.sympy(
        index=new_attraction_index
    )
)

new_attraction_table.head(10)

In [ ]:
candidate_indices = [
    int(index)
    for index in new_attraction_table["equation_index"].head(3)
]

attraction_candidates = [
    {
        "name": "old_attraction",
        "model": regime_models["attraction_dominated"],
        "index": selected_regime_indices["attraction_dominated"]
    }
]

attraction_candidates.extend(
    {
        "name": f"expanded_index_{index}",
        "model": attraction_model_expanded,
        "index": index
    }
    for index in candidate_indices
)

candidate_results = []

for dataset, dataset_name in [
    (
        validation_df[
            validation_df["regime"]
            == "attraction_dominated"
        ],
        "natural_validation"
    ),
    (
        stress_validation_df[
            stress_validation_df["regime"]
            == "attraction_dominated"
        ],
        "stress_validation"
    )
]:
    X = dataset[FEATURE_COLUMNS].to_numpy()

    for candidate in attraction_candidates:
        predictions = candidate["model"].predict(
            X,
            index=candidate["index"]
        )

        candidate_results.append(
            evaluate_predictions(
                dataset,
                predictions,
                dataset_name,
                candidate["name"]
            )
        )

attraction_candidate_comparison = pd.DataFrame(
    candidate_results
)

attraction_candidate_comparison[
    [
        "model",
        "dataset",
        "delta_Z_RMSE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]


In [ ]:
final_regime_models = regime_models.copy()
final_regime_indices = selected_regime_indices.copy()

final_regime_models[
    "attraction_dominated"
] = attraction_model_expanded

# Use the index selected from natural + stress validation.
final_regime_indices[
    "attraction_dominated"
] = int(new_attraction_index)

print(
    "Final attraction equation index:",
    final_regime_indices["attraction_dominated"]
)
print("Final attraction equation:")
print(
    attraction_model_expanded.sympy(
        index=final_regime_indices["attraction_dominated"]
    )
)


In [ ]:
final_validation_results = []

for dataset, dataset_name in [
    (
        validation_df,
        "natural_validation"
    ),
    (
        stress_validation_df,
        "stress_validation"
    )
]:
    predictions = predict_selected_regime_models(
        final_regime_models,
        final_regime_indices,
        dataset
    )

    final_validation_results.append(
        evaluate_predictions(
            dataset,
            predictions,
            dataset_name,
            "improved_oracle_regime_specific"
        )
    )

final_oracle_results = pd.DataFrame(
    final_validation_results
)

final_oracle_results

In [ ]:
final_model_comparison = pd.concat([
    updated_model_comparison,
    final_oracle_results
], ignore_index=True)

final_model_comparison[
    [
        "model",
        "dataset",
        "delta_Z_RMSE",
        "delta_Z_MAE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]

In [ ]:
print(
    "Global index:",
    selected_global_index
)

print(
    "Global equation:",
    global_model.sympy(
        index=selected_global_index
    )
)

print("\nFinal regime indices:")

for regime_name in REGIME_NAMES:
    print(
        regime_name,
        final_regime_indices[regime_name]
    )

    print(
        final_regime_models[
            regime_name
        ].sympy(
            index=final_regime_indices[
                regime_name
            ]
        )
    )

print("\nFinal comparison:")
display(final_model_comparison)

In [ ]:
final_model_comparison.to_csv(
    TABLE_DIR / "final_model_comparison.csv",
    index=False
)

attraction_candidate_comparison.to_csv(
    TABLE_DIR / "attraction_candidate_comparison.csv",
    index=False
)

with open(MODEL_DIR / "final_regime_models.pkl", "wb") as file:
    pickle.dump(final_regime_models, file)

with open(MODEL_DIR / "final_regime_indices.json", "w") as file:
    json.dump(
        {
            name: int(index)
            for name, index in final_regime_indices.items()
        },
        file,
        indent=2
    )

print("Final regime models and selections saved to:", RESULTS_DIR)
